In [1]:
import os

NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
    with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
        f.write("""{
        "file_format_version" : "1.0.0",
        "ICD" : {
            "library_path" : "libEGL_nvidia.so.0"
        }
    }
    """)
        
os.environ["MUJOCO_GL"] = "egl"

import torch
from ddpg_trainer import  Trainer
from RLAlg.alg.ddpg_double_q import DDPGDoubleQ

In [2]:
task_name = "cheetah"
seed = 0

In [3]:
trainer = Trainer(task_name, seed)

In [4]:
%%timeit
trainer.rollout()

311 ms ± 388 μs per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [5]:
%%timeit
trainer.update(100, 1000)

1.14 s ± 24.8 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [6]:
%%timeit
batch = trainer.replay_buffer.sample(1000)
obs_batch = batch["states"].to(trainer.device)
action_batch = batch["actions"].to(trainer.device)
reward_batch = batch["rewards"].to(trainer.device)
done_batch = batch["dones"].to(trainer.device)
next_obs_batch = batch["next_states"].to(trainer.device)

194 μs ± 2.54 μs per loop (mean ± std. dev. of 7 runs, 10,000 loops each)


In [7]:
batch = trainer.replay_buffer.sample(1000)
obs_batch = batch["states"].to(trainer.device)
action_batch = batch["actions"].to(trainer.device)
reward_batch = batch["rewards"].to(trainer.device)
done_batch = batch["dones"].to(trainer.device)
next_obs_batch = batch["next_states"].to(trainer.device)

In [8]:
%%timeit
feature_batch = trainer.encoder(obs_batch, True)
with torch.no_grad():
    next_feature_batch = trainer.encoder(next_obs_batch, True)

trainer.encoder_optimizer.zero_grad(set_to_none=True)
trainer.critic_optimizer.zero_grad(set_to_none=True)
critic_loss = DDPGDoubleQ.compute_critic_loss(
    trainer.actor, trainer.critic, trainer.critic_target, feature_batch, action_batch, reward_batch, next_feature_batch, done_batch, trainer.std, trainer.gamma
)
critic_loss.backward()
trainer.critic_optimizer.step()
trainer.encoder_optimizer.step()

4.21 ms ± 6.44 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)


In [9]:
!pip install line_profiler

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 750.2/750.2 kB 8.1 MB/s eta 0:00:00


In [10]:
%load_ext line_profiler

In [11]:
%lprun -f trainer.update trainer.update(100, 1000)

Timer unit: 1e-09 s

Total time: 0.932172 s
File: /home/xd/Desktop/SEVEA/ddpg_trainer.py
Function: update at line 162

Line #      Hits         Time  Per Hit   % Time  Line Contents
   162                                               def update(self, num_iteration:int, batch_size:int):
   163       101      32106.0    317.9      0.0          for _ in range(num_iteration):
   164       100   17828573.0 178285.7      1.9              batch = self.replay_buffer.sample(batch_size)
   165       100   15066264.0 150662.6      1.6              obs_batch = batch["states"].to(self.device)
   166       100    3112916.0  31129.2      0.3              action_batch = batch["actions"].to(self.device)
   167       100    1595639.0  15956.4      0.2              reward_batch = batch["rewards"].to(self.device)
   168       100    1628031.0  16280.3      0.2              done_batch = batch["dones"].to(self.device)
   169       100    2206009.0  22060.1      0.2              next_obs_batch = batch["next